In [1]:
import pandas as pd

In [114]:
# read in ssm data

ssm = pd.read_excel('./ssm_model_obs_pair.xlsx', sheet_name='Sheet1')

In [3]:
print(ssm)

       Station          Model_Time  Nodes   ID Type  Depth_m Masked_Area?  \
0       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.0           No   
1       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      1.5           No   
2       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.0           No   
3       ADM001 2014-03-13 12:00:00   6231  MMU  Lab      1.5           No   
4       ADM001 2014-03-13 12:00:00   6231  MMU  CTD      2.5           No   
...        ...                 ...    ...  ...  ...      ...          ...   
101239  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.0           No   
101240  FID001 2014-11-18 12:00:00   5830  MMU  CTD      8.5           No   
101241  FID001 2014-11-18 12:00:00   5830  MMU  CTD      9.0           No   
101242  FID001 2014-11-18 12:00:00   5830  MMU  CTD     11.5           No   
101243  FID001 2014-11-18 12:00:00   5830  MMU  CTD     12.0           No   

        Temp_C  Salinity_psu  DO_mgL  ...  Model_DO_mgL  Model_NO23N_mgL  \

In [13]:
ssm.columns

Index(['name', 'time', 'Nodes', 'ID', 'Type', 'z', 'Masked_Area?', 'Temp_C',
       'Salinity_psu', 'DO_mgL', 'Chla_ugL', 'NO23N_mgL', 'NH4N_mgL',
       'PAR_Em2day', 'Layer', 'CT', 'SA', 'DO', 'NO3', 'NH4',
       'Model_PAR_Em2day', 'Chl', 'Layer_Depth_m', 'Embayment', 'Layer_Cat',
       'Hydro_T', 'WQM_T'],
      dtype='object')

In [115]:
# read in 2014 lo_ssc data
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc.pkl")

In [26]:
print(data["obs"])

         cid         lon        lat                time          z         SA  \
0        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.148984   
1        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149086   
2        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149387   
3        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -1.586480  31.149889   
4        0.0 -123.248337  48.618332 2014-02-11 06:35:53  -5.255167  31.145790   
...      ...         ...        ...                 ...        ...        ...   
5013  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -24.400000        NaN   
5014  3350.0 -122.428001  47.744000 2014-12-15 17:41:00 -33.900000        NaN   
5015  3351.0 -122.428001  47.744000 2014-12-15 17:42:00 -14.900000        NaN   
5016  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -0.620000        NaN   
5017  3352.0 -122.428001  47.744000 2014-12-15 17:43:00  -1.600000        NaN   

            CT          DO 

In [16]:
data["obs"].columns

Index(['cid', 'lon', 'lat', 'time', 'z', 'SA', 'CT', 'DO', 'NO3', 'Chl',
       'name', 'cruise', 'source', 'NH4', 'PO4 (uM)', 'SiO4 (uM)', 'NO2 (uM)',
       'TA', 'DIC'],
      dtype='object')

In [116]:
# rename ssm columns to match the other dataframes

ssm = ssm.rename(columns={
    'Station': 'name',
    'Depth_m': 'z',
    'Model_Time': 'time',
    'Model_Temp_C': 'CT',
    'Model_Salinity_psu': 'SA',
    'Model_DO_mgL': 'DO',
    'Model_Chla_ugL': 'Chl',
    'Model_NO23N_mgL': 'NO3',
    'Model_NH4N_mgL': 'NH4'}) 

# convert DO from mg/L to umol/L 
ssm["DO"] = ssm["DO"] * 1000 / 31.998 

# convert NH4 from mg/L to umol/L 
ssm["NH4"] = ssm["NH4"] * 1000 / 18.039 

# convert NO3 from mg/L to umol/L 
ssm["NO3"] = ssm["NO3"] * 1000 / 62.004 

# make depth negative 
ssm["z"] = -ssm["z"] 

# ensure datetime is in datetime format 
ssm["time"] = pd.to_datetime(ssm["time"]) 

# standardize depth to float 
ssm["z"] = ssm["z"].astype(float)

In [117]:
# copies
obs = data['obs'].copy()
ssm_df = ssm.copy()

# formats
obs['time'] = pd.to_datetime(obs['time'])
ssm_df['time'] = pd.to_datetime(ssm_df['time'])

obs['z'] = obs['z'].round(2)
ssm_df['z'] = ssm_df['z'].round(2)

# marker indicating a successful obs match
obs_match = obs[['cid','name', 'z', 'time']].copy()
obs_match['matched_obs'] = True

# preserve index
obs_match= obs_match.reset_index().rename(columns={'index':'obs_index'})

# sort (required)
obs_match = obs_match.sort_values('time')
ssm_df = ssm_df.sort_values('time')

# asof merge with tolerance
ssm_matched = pd.merge_asof(
    obs_match,
    ssm_df,
    on='time',
    by=['name','z'],
    direction='nearest',
    tolerance=pd.Timedelta('3hr')
)

# keep only successful ssm matches
ssm_vars = ['CT', 'SA', 'DO', 'NO3', 'NH4', 'Chl']
ssm_matched = ssm_matched.dropna(subset=ssm_vars)

In [118]:
# keep only standardized columns
keep_cols = [
    'obs_index','cid','name', 'time', 'z',
    'CT', 'SA', 'DO', 'NO3', 'NH4', 'Chl'
]

ssm_matched = ssm_matched[keep_cols]

In [97]:
print(ssm_matched)
ssm_matched.columns

         cid    name       z                time  matched_obs    Nodes  ID  \
61    1849.0  CK200P  -24.70 2014-01-21 08:44:00         True   9228.0  KC   
62    1849.0  CK200P  -34.60 2014-01-21 08:44:00         True   9228.0  KC   
63    1849.0  CK200P  -14.70 2014-01-21 08:45:00         True   9228.0  KC   
64    1849.0  CK200P   -0.96 2014-01-21 08:46:00         True   9228.0  KC   
65    1843.0  KSBP01 -197.00 2014-01-21 09:25:00         True   8908.0  KC   
...      ...     ...     ...                 ...          ...      ...  ..   
5011  1229.0  NSEX01  -34.80 2014-12-16 12:38:00         True  12735.0  KC   
5012  1229.0  NSEX01  -24.90 2014-12-16 12:39:00         True  12735.0  KC   
5013  1229.0  NSEX01  -14.90 2014-12-16 12:40:00         True  12735.0  KC   
5014  1229.0  NSEX01   -0.83 2014-12-16 12:41:00         True  12735.0  KC   
5015  1229.0  NSEX01   -1.80 2014-12-16 12:41:00         True  12735.0  KC   

     Type Masked_Area?  Temp_C  ...          DO       NO3      

Index(['cid', 'name', 'z', 'time', 'matched_obs', 'Nodes', 'ID', 'Type',
       'Masked_Area?', 'Temp_C', 'Salinity_psu', 'DO_mgL', 'Chla_ugL',
       'NO23N_mgL', 'NH4N_mgL', 'PAR_Em2day', 'Layer', 'CT', 'SA', 'DO', 'NO3',
       'NH4', 'Model_PAR_Em2day', 'Chl', 'Layer_Depth_m', 'Embayment',
       'Layer_Cat', 'Hydro_T', 'WQM_T'],
      dtype='object')

In [121]:
keep_df = ssm_matched[['obs_index']].drop_duplicates()

for k in ['obs', 'cas7_t1_x11ab', 'ssc']:
    data[k] = (
        data[k]
        .reset_index().rename(columns={'index': 'obs_index'})
        .merge(keep_df, on='obs_index', how='inner')
        .drop(columns=['obs_index'])
        .reset_index(drop=True)
    )

data['ssm'] = ssm_matched.reset_index(drop=True)

In [ ]:
print(ssm_matched)
ssm_matched.columns

         cid    name       z                time  matched_obs    Nodes  ID  \
61    1849.0  CK200P  -24.70 2014-01-21 08:44:00         True   9228.0  KC   
62    1849.0  CK200P  -34.60 2014-01-21 08:44:00         True   9228.0  KC   
63    1849.0  CK200P  -14.70 2014-01-21 08:45:00         True   9228.0  KC   
64    1849.0  CK200P   -0.96 2014-01-21 08:46:00         True   9228.0  KC   
65    1843.0  KSBP01 -197.00 2014-01-21 09:25:00         True   8908.0  KC   
...      ...     ...     ...                 ...          ...      ...  ..   
5011  1229.0  NSEX01  -34.80 2014-12-16 12:38:00         True  12735.0  KC   
5012  1229.0  NSEX01  -24.90 2014-12-16 12:39:00         True  12735.0  KC   
5013  1229.0  NSEX01  -14.90 2014-12-16 12:40:00         True  12735.0  KC   
5014  1229.0  NSEX01   -0.83 2014-12-16 12:41:00         True  12735.0  KC   
5015  1229.0  NSEX01   -1.80 2014-12-16 12:41:00         True  12735.0  KC   

     Type Masked_Area?  Temp_C  ...          DO       NO3      

Index(['cid', 'name', 'z', 'time', 'matched_obs', 'Nodes', 'ID', 'Type',
       'Masked_Area?', 'Temp_C', 'Salinity_psu', 'DO_mgL', 'Chla_ugL',
       'NO23N_mgL', 'NH4N_mgL', 'PAR_Em2day', 'Layer', 'CT', 'SA', 'DO', 'NO3',
       'NH4', 'Model_PAR_Em2day', 'Chl', 'Layer_Depth_m', 'Embayment',
       'Layer_Cat', 'Hydro_T', 'WQM_T'],
      dtype='object')

In [122]:
for k in ['obs', 'cas7_t1_x11ab', 'ssc', 'ssm']:
    print(k, len(data[k]), data[k].index[:5])

obs 1077 RangeIndex(start=0, stop=5, step=1)
cas7_t1_x11ab 1077 RangeIndex(start=0, stop=5, step=1)
ssc 1077 RangeIndex(start=0, stop=5, step=1)
ssm 1077 RangeIndex(start=0, stop=5, step=1)


In [123]:
print(data['ssm'])

      obs_index     cid    name                time       z         CT  \
0          4055  1849.0  CK200P 2014-01-21 08:44:00  -24.70   9.770035   
1          4056  1849.0  CK200P 2014-01-21 08:44:00  -34.60   9.716570   
2          4057  1849.0  CK200P 2014-01-21 08:45:00  -14.70   9.577158   
3          4058  1849.0  CK200P 2014-01-21 08:46:00   -0.96   8.161728   
4          4036  1843.0  KSBP01 2014-01-21 09:25:00 -197.00   9.047412   
...         ...     ...     ...                 ...     ...        ...   
1072       4023  1229.0  NSEX01 2014-12-16 12:38:00  -34.80  10.964799   
1073       4024  1229.0  NSEX01 2014-12-16 12:39:00  -24.90  10.958919   
1074       4025  1229.0  NSEX01 2014-12-16 12:40:00  -14.90  10.921903   
1075       4026  1229.0  NSEX01 2014-12-16 12:41:00   -0.83   9.138308   
1076       4027  1229.0  NSEX01 2014-12-16 12:41:00   -1.80   9.138308   

             SA          DO       NO3       NH4       Chl  
0     29.604378  243.519118  7.171316  1.034263  0.

In [124]:
# save the combined data
pd.to_pickle(data, 'combined_bottle_2014_cas7_t1_x11ab_ssc_ssm.pkl')

In [125]:
data = pd.read_pickle("./combined_bottle_2014_cas7_t1_x11ab_ssc_ssm.pkl")

In [126]:
for k in data:
    print(k, len(data[k]))

obs 1077
cas7_t1_x11ab 1077
ssc 1077
meta 2
ssm 1077
